<a href="https://colab.research.google.com/github/Saisneha0209/Machine-Learning/blob/main/AI_Loan_Risk_Evaluater.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install langchain-community langchain-core langchain-mistralai pypdf -q

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from getpass import getpass

from langchain_core.embeddings import Embeddings
from langchain_mistralai.embeddings import MistralAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from google.colab import files

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier

os.environ["MISTRAL_API_KEY"] = getpass("Enter your Mistral AI API Key: ")

In [5]:
#First we will process our document!!
def process_document():
    print("Please upload your mock Bank Statement PDF...")
    uploaded = files.upload()
    if not uploaded:
        print("No file selected!")
        return None, None

    pdf_filename = list(uploaded.keys())[0]
    loader = PyPDFLoader(pdf_filename)
    docs = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )
    split_chunks = text_splitter.split_documents(docs)
    print(f"\nDocument processed into {len(split_chunks)} semantic chunks.")

    embeddings1 = MistralAIEmbeddings(model="mistral-embed")
    raw_texts = [chunk.page_content for chunk in split_chunks]

    print("Generating vector embeddings via Mistral AI...")
    embeddings = embeddings1.embed_documents(raw_texts)
    print(f"Generated vectors with dimension size: {len(embeddings[0])}")

    return split_chunks, np.array(embeddings)

#Now we will work upon to detect for the risk management!!

def risk(embeddings):
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    labels = kmeans.fit_predict(embeddings)

    cluster = pd.Series(labels).mode()[0]

    risk_mapping = {0: "Low Risk", 1: "Medium Risk", 2: "High Risk"}
    assign = risk_mapping[cluster]

    pca = PCA(n_components=2)
    reduce_vectors = pca.fit_transform(embeddings)
    plt.figure(figsize=(8, 5))
    sns.scatterplot(
        x=reduce_vectors[:, 0],
        y=reduce_vectors[:, 1],
        hue=[risk_mapping[c] for c in labels],
        palette='Set1', s=100
    )
    plt.title("Statement Analysis (Clusters)")
    plt.xlabel("PCA Component 1")
    plt.ylabel("PCA Component 2")
    plt.grid(True)
    plt.show()

    return assign

#Now we will watch out for the loan managment!!
def loan_application(risk, annual_income, monthly_debt):
    group = {"Low Risk": 0, "Medium Risk": 1, "High Risk": 2}
    group1 = group[risk]

    X_train = np.array([
        [120000, 0.15, 0],  # Approved
        [30000,  0.55, 2],  # Denied
        [75000,  0.30, 1],  # Approved
        [50000,  0.45, 2],  # Denied
        [90000,  0.20, 0],  # Approved
    ])  #Annual in,DTI RATIO,RISK CATEGORY!!
    y_train = np.array([1, 0, 1, 0, 1])
    cl = DecisionTreeClassifier(random_state=42)
    cl.fit(X_train, y_train)

    dti_ratio = monthly_debt / (annual_income / 12)
    applicant_features = np.array([[annual_income, dti_ratio, group1]])

    prediction = cl.predict(applicant_features)[0]
    decision = "APPROVED" if prediction == 1 else "REJECTED"

    print("\n" + "="*40)
    print("           AI DECISION")
    print("="*40)
    print(f"Current Risk:     {risk}")
    print(f"Calculated Ratio:   {dti_ratio:.2%}")
    print(f"Final Decision:     {decision}")
    print("="*40)

In [ ]:
chunks, embeddings_matrix = process_document()

if embeddings_matrix is not None:
    detect_risk = risk(embeddings_matrix)
    income = 85000
    monthly_debt = 1200

    loan_application(detect_risk, income, monthly_debt)

#USING TENSORFLOW

In [7]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.preprocessing import StandardScaler

In [8]:
def loan_application(risk_tier, annual_income, monthly_debt):
    group = {"Low Risk": 0, "Medium Risk": 1, "High Risk": 2}
    group1 = group[risk_tier]

    X_train = np.array([
        [120000, 0.15, 0],
        [30000,  0.55, 2],
        [75000,  0.30, 1],
        [50000,  0.45, 2],
        [90000,  0.20, 0],
    ], dtype=np.float32)

    y_train = np.array([1, 0, 1, 0, 1], dtype=np.float32)

    dti_ratio = monthly_debt / (annual_income / 12)
    applicant_features = np.array([[annual_income, dti_ratio, group1]], dtype=np.float32)

  #nOW WE WILL SCALE THE DAT as neural network requires scaling of data!!
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    applicant_scaled = scaler.transform(applicant_features)

    #Now we will build TensorFlow Neural Network
    model = Sequential([
        #Here we have 8 neurons,and will looke for basic correlations...
        Dense(8, activation='relu', input_shape=(3,)),
        #Here we have 4 neurons, and it will capture mathematical patterns...
        Dense(4, activation='relu'),
        #IN THE OUTPUT LAYER ,1 neuron with Sigmoid to give a probability between 0 and 1...
        Dense(1, activation='sigmoid')
    ])

    #Now gonna Compile the Model...
    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    # 6. Train the Neural Network...
    print("Training TensorFlow Neural Network...")
    model.fit(X_train_scaled, y_train, epochs=10)
    prediction = model.predict(applicant_scaled)[0][0]

    #If probability is > 50%, approve the loan...
    decision = "APPROVED" if prediction >= 0.5 else "REJECTED"

    print("\n" + "="*40)
    print("       TENSORFLOW AI DECISION")
    print("="*40)
    print(f"Extracted Risk:     {risk_tier}")
    print(f"Calculated Ratio:   {dti_ratio:.2%}")
    print(f"Approval Confidence: {prediction:.2%}")
    print(f"Final Decision:     {decision}")
    print("="*40)

In [ ]:
chunks, embeddings_matrix = process_document()

if embeddings_matrix is not None:
    detect_risk = risk(embeddings_matrix)
    income = 85000
    monthly_debt = 1200

    loan_application(detect_risk, income, monthly_debt)

In [ ]:
!pip install chromadb

In [ ]:
!pip install --upgrade mistralai

In [ ]:
import chromadb
from mistralai.client import Mistral
client = chromadb.Client()
collection = client.get_or_create_collection("text_collection")
raw_text_chunks = [chunk.page_content for chunk in chunks]

print("ChromaDB Vector Store...")
for i, Text in enumerate(raw_text_chunks):
  collection.add(
           documents=[Text],
            metadatas=[{"chunk_id": i}],
            ids=[str(i)]
        )
print("VECTOR - DB HAS BEEN CREATED SUCCESSFULLY!!!")

query = "What are the main monthly income sources and transaction patterns?"
print(f"\nQuerying Database: '{query}'")

results = collection.query(
        query_texts=[query],
        n_results=5
    )
result_text = ""
for i in range(len(results['documents'][0])):
  result_text += results['documents'][0][i] + "\n"

instruction = """You are a data answering assistant. Your job is to answer the questions based on the content that these chunks are saying.
Rely only on clear factual answers. Do not assume anything!!
And do not bring in any outside knowledge.
"""
final_question = instruction + "\nQuestion: " + query + "\n\nContext:\n" + result_text
MISTRAL_API_KEY = os.environ.get("MISTRAL_API_KEY")
mistral_client = Mistral(api_key=MISTRAL_API_KEY)
response = mistral_client.chat.complete(
        model="mistral-large-latest",
        messages=[{"role": "user", "content": final_question}]
    )

print("\n" + "="*40)
print("        MISTRAL DATA ANSWERS")
print("="*40)
print(response.choices[0].message.content)
print("="*40)